# Lab 9b: Sequence Labelling with BERT
In this bonus lab, we will look at sequence labelling with BERT.

It is heavily based on the huggingface tutorial: https://huggingface.co/learn/nlp-course/en/chapter7/2

## Data Preparation

First of all we need a dataset suitable for sequence labelling.  Heere we use the CoNLL-2003 dataset, which contains news stories from Reuters which have been annotated for part-of-speech and named entities.

In [1]:
#needed on colab
#!pip install datasets

In [3]:
import datasets
print(datasets.config.HF_DATASETS_CACHE)

/Users/lukebirkett/.cache/huggingface/datasets


In [4]:
from datasets import load_dataset

# We point to the processed version that has NO .py script
raw_datasets = load_dataset("lhoestq/conll2003")

# If that still hits the local cache, try the specific repo path:
# raw_datasets = load_dataset("huggingface/conll2003")

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


Inspecting the raw_datasets object shows is the columns present and the split between training, validation and set.

In [5]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Let's have a look at the first element of the training set in terms of tokens and ner_tags.

In [6]:
raw_datasets["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [7]:
raw_datasets["train"][0]["tokens"]

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

In [8]:
raw_datasets["train"][0]["ner_tags"]

[3, 0, 7, 0, 0, 0, 7, 0, 0]

To see what these integer labels actually mean, we need to refer to the features of the dataset and their names

In [9]:
ner_feature = raw_datasets["train"].features["ner_tags"]
label_names=ner_feature.feature.names
label_names

AttributeError: 'Value' object has no attribute 'names'

In [10]:
from datasets import Features, Sequence, ClassLabel, Value

# Define the features structure manually
features = raw_datasets["train"].features.copy()
features["ner_tags"] = Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

# Apply the new features to the dataset
raw_datasets = raw_datasets.cast(features)

# Now your original code will work!
ner_feature = raw_datasets["train"].features["ner_tags"]
label_names = ner_feature.feature.names
print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


### Exercise 1
Write a function to which takes an index to the training data  as input and displays the corresponding sentence tokens and label_names as in this example:

```
display_labels(index=0)
--------------------------
EU : B-ORG
rejects : O
German : B-MISC
call : O
to : O
boycott : O
British : B-MISC
lamb : O
. : O
```


In [11]:
def display_labels(index=0):
    words=raw_datasets["train"][index]["tokens"]
    labels=raw_datasets["train"][index]["ner_tags"]

    for word, label in zip(words,labels):
        line="{} : {}".format(word,label_names[label])
        print(line)

display_labels(3)

The : O
European : B-ORG
Commission : I-ORG
said : O
on : O
Thursday : O
it : O
disagreed : O
with : O
German : B-MISC
advice : O
to : O
consumers : O
to : O
shun : O
British : B-MISC
lamb : O
until : O
scientists : O
determine : O
whether : O
mad : O
cow : O
disease : O
can : O
be : O
transmitted : O
to : O
sheep : O
. : O


## Tokenisation

Now we are going to need to tokenize the text and then align the labels.  
We need to use the standard BERT tokenizer but tell it that the text is already split into words.  It will then carry out wordpiece tokenization.  We then need to align the labels - which requires adding labels where a word in the original text has been split into multiple wordpieces.

In [12]:
from transformers import AutoTokenizer
model_checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)

In [13]:
#lets try the tokenizer on different lines of the training data
#find a line where at least one word is split into subwords
inputs=tokenizer(raw_datasets["train"][3]["tokens"],is_split_into_words=True)
inputs.tokens()

['[CLS]',
 'the',
 'european',
 'commission',
 'said',
 'on',
 'thursday',
 'it',
 'disagreed',
 'with',
 'german',
 'advice',
 'to',
 'consumers',
 'to',
 'shu',
 '##n',
 'british',
 'lamb',
 'until',
 'scientists',
 'determine',
 'whether',
 'mad',
 'cow',
 'disease',
 'can',
 'be',
 'transmitted',
 'to',
 'sheep',
 '.',
 '[SEP]']

If we look at the word_ids for this sentence, we can see that special tokens (CLS and SEP) do not have a word_id (as they are not in the vocabulary).  A word which has been split into multiple subwords has the same word_id for each subword.

In [14]:
inputs.word_ids()

[None,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 None]

### Exercise 2

We need a function to align the labels with the new tokens.  In particular, we need to use a special label '-100'  for any tokens which are not in the vocabulary (i.e., don't have a word_id) and create the label for subsequent parts of a word. If the label is B-XXX then the subsequenct label needs to be I-XXX.  Make sure you understand the following function which takes a list of labels and a list of word_ids and returns a new *aligned* list of labels.  Test it out on some different inputs from the training data

Standard NER datasets come with one label per word (e.g., "London" is labeled B-LOC). However, BERT uses WordPiece tokenization, which often breaks a single word into multiple sub-tokens (e.g., "Gutenberg" becomes ["Guten", "##berg"]).

**The Problem:** If you have 10 words and 10 labels, but BERT tokenizes those into 14 pieces, your lists no longer line up. The model won't know which label goes with the "##berg" part.

In [15]:
def align_labels_with_tokens(labels,word_ids):
    new_labels=[]
    current_word = ''
    for word_id in word_ids:
        if word_id ==current_word:
            #same word as previous token
            label = labels[word_id]
            #update B-XXX to I-XXX
            if label % 2==1:
                label+=1
        else:
            #start of new word
            current_word=word_id
            if word_id is None:
                #special token
                label=-100

            else:
                label=labels[word_id]

        new_labels.append(label)

    return new_labels

In [16]:
labels = raw_datasets["train"][3]["ner_tags"]
inputs=tokenizer(raw_datasets["train"][3]["tokens"],is_split_into_words=True)
word_ids = inputs.word_ids()
print("ner labels:", labels)
print("input ids:", inputs)
print("word ids:", word_ids)
print("aligned:", align_labels_with_tokens(labels, word_ids))


ner labels: [0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
input ids: {'input_ids': [101, 1996, 2647, 3222, 2056, 2006, 9432, 2009, 18335, 2007, 2446, 6040, 2000, 10390, 2000, 18454, 2078, 2329, 12559, 2127, 6529, 5646, 3251, 5506, 11190, 4295, 2064, 2022, 11860, 2000, 8351, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
word ids: [None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, None]
aligned: [-100, 0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]


We can now apply this to the whole dataset

In [17]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [18]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

In [19]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [20]:
tokenized_datasets["train"][0]["input_ids"]

[101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102]

In [21]:
tokenized_datasets["train"][0]["labels"]

[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, -100]

## Adding padding

We need to make sure all of the inputs in a batch are the same length.  We do this by adding padding using the DataCollatorForTokenClassification class.

Padding is necessary because machine learning models (and GPUs in particular) are designed to process data in batches rather than one sentence at a time. To do this efficiently, the data must be structured as a uniform, rectangular matrix.

Also, most BERT implementations are configured with a max_position_embeddings value (typically 512). While BERT can handle up to 512 tokens, the underlying math of the "Position Embeddings" requires a fixed reference point.By padding a 20-word sentence to 512, you ensure the model knows exactly which position index to assign to each word, maintaining a consistent structure for the self-attention mechanism.

Since [PAD] tokens don't mean anything, we don't want the model to waste time with them. To overcome this we create an Attention Mask alongside padding. The mask is a list of 1s and 0s where 1 This is a real word and 0 is just padding.

When you see padding=True in the tokenizer, it is automatically adding those 0s and generating the mask so that the [PAD] tokens don't interfere with the Named Entity Recognition scores.

In [22]:
from transformers import DataCollatorForTokenClassification
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

_pre_pad = [tokenized_datasets["train"][i] for i in range(2)]
print(_pre_pad[0]["labels"])
print(_pre_pad[1]["labels"])

#let's see what this looks like for a batch of size 2 (the first 2 items in the training data)
batch=data_collator(_pre_pad)
batch["labels"]

[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, -100]
[-100, 1, 2, -100]


tensor([[-100,    3,    0,    7,    0,    0,    0,    7,    0,    0, -100],
        [-100,    1,    2, -100, -100, -100, -100, -100, -100, -100, -100]])

## Evaluation Metrics
We need to be able to evaluate how good the predictions of a model are.

### Exercise 3
Write a function compute_accuracy which will take predictions and references (as lists of lists of labels) and return the accuracy.

For example:

```
compute_accuracy([['O','O','B-PER','O'],['B-MISC','I-MISC','O']],[['O','O','B-PER','O'],['B-MISC','I-MISC','O']])
-----------------
0.857
```

In [23]:
def compute_accuracy(predictions,references):
    #predictions and references are each lists of lists of labels for comparison

    count=0
    correct=0
    for preds,refs in zip (predictions,references):
        for pred,ref in zip(preds,refs):
            count+=1
            if pred==ref:
                correct+=1
    return float(correct)/float(count)



test_preds=[['O','O','B-PER','O'],['B-MISC','I-MISC','O']]
test_refs=[['O','O','B-ORG','O'],['B-MISC','I-MISC','O']]
compute_accuracy(test_preds,test_refs)





0.8571428571428571

Now we are going to apply this to the logits which will come back from the model.  We need to select the most likely prediction for each token using argmax and then strip away special tokens before passing the lists of lists of labels to the compute_accuracy function just defined.

In future, we could define (or import) a greater range of evaluation metrics (e.g., precision and recall for each of the different labels), but here we are just going to use overall accuracy as an indicator of how well our model might be doing.  You might want to check out the seqeval library

In [24]:

label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [25]:
import numpy as np

def compute_metrics(eval_preds):
    #eval_preds is a batch of logits and labels (list of lists)
    logits,labels = eval_preds
    predictions=np.argmax(logits, axis=-1)

    #remove ignored index (special tokens) and convert to labels
    true_labels=[[label_names[l] for l in label if l!=-100] for label in labels]
    true_predictions = [[label_names[p] for (p,l) in zip(prediction,label) if l!=-100] for prediction,label in zip(predictions,labels)]

    #we could compute other metrics as well, but here, for simplicity we are just going to compute overall accuracy
    accuracy = compute_accuracy(predictions=true_predictions,references=true_labels)
    return {"overall accuracy":accuracy}


## Defining the model

First of all we need to make sure we have 2 dictionaries which define the correspondance between ids and labels

In [26]:
id2label={i:label for i,label in enumerate(label_names)}
label2id={v:k for k,v in id2label.items()}

print(id2label)
print(label2id)

{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


In [27]:
model_checkpoint

'bert-base-uncased'

Now we can set up the pre-trained model ready to be used for sequence labelling / token classification

In [28]:
from transformers import AutoModelForTokenClassification

model =AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 17288.32it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; n

In [29]:
model.config.num_labels

9

In [30]:
#I cannot run this on my laptop.  But if I train the model on colab, I can save it, download it and then run it on my laptop
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

In [34]:
import torch

# Check if CUDA (NVIDIA GPU) is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Check for NVIDIA GPU (cuda) or Apple Silicon GPU (mps)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu
Using device: mps


In [35]:
#make sure you have changed the runtype to GPU on colab!

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Overall accuracy
1,0.071139,0.054203,0.985718
2,0.031386,0.053753,0.987084
3,0.017600,0.055356,0.987561


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.51it/s]
/Users/lukebirkett/Repos/study-planner/968G5_Advanced_Natural_Language_Processing/adv_nlp/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.29it/s]
/Users/lukebirkett/Repos/study-planner/968G5_Advanced_Natural_Language_Processing/adv_nlp/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]


TrainOutput(global_step=5268, training_loss=0.043188904077606664, metrics={'train_runtime': 1102.1558, 'train_samples_per_second': 38.219, 'train_steps_per_second': 4.78, 'total_flos': 891900038010780.0, 'train_loss': 0.043188904077606664, 'epoch': 3.0})

In [37]:
#once the model has been saved, it can be run independently elsewhere
trainer.save_model("julie-ner")


## Using the Model

The saved model can now be used in a number of ways.  It can be used as part of a "token-classification" pipeline which automatically carries out the appropriate tokenization on the input and then runs the model

In [27]:
#run this without training (or running any of the previous cells) if the model is saved (and available)

from transformers import pipeline

ner_classifier=pipeline("token-classification",model="julie-ner")

sentence="Germans shun British lamb"
ner_classifier(sentence)

[{'entity': 'B-MISC',
  'score': 0.997512,
  'index': 1,
  'word': 'germans',
  'start': 0,
  'end': 7},
 {'entity': 'B-MISC',
  'score': 0.9993293,
  'index': 4,
  'word': 'british',
  'start': 13,
  'end': 20}]

Alternatively, the model can be run directly on tokenized inputs, such as we have already prepared for our test data.

The cells below will remind us what those datasets look like - both in raw and tokenized forms.

In [28]:
raw_datasets["test"][0]

{'id': '0',
 'tokens': ['SOCCER',
  '-',
  'JAPAN',
  'GET',
  'LUCKY',
  'WIN',
  ',',
  'CHINA',
  'IN',
  'SURPRISE',
  'DEFEAT',
  '.'],
 'pos_tags': [21, 8, 22, 37, 22, 22, 6, 22, 15, 12, 21, 7],
 'chunk_tags': [11, 0, 11, 21, 11, 12, 0, 11, 13, 11, 12, 0],
 'ner_tags': [0, 0, 5, 0, 0, 0, 0, 1, 0, 0, 0, 0]}

In [29]:
tokenized_datasets["test"][0]

{'input_ids': [101,
  4715,
  1011,
  2900,
  2131,
  5341,
  2663,
  1010,
  2859,
  1999,
  4474,
  4154,
  1012,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 0, 0, 5, 0, 0, 0, 0, 1, 0, 0, 0, 0, -100]}

We can instantiate an AutoModelForTokenClassification model using the saved checkpoint.

In [30]:
from transformers import AutoModelForTokenClassification
model_checkpoint="julie-ner"
model =AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id)

Now we need to wrap the attention_mask and input_ids as tensors and pass that to the model.


In [31]:
import torch
test_input=tokenized_datasets["test"][0]
mask_tensor=torch.tensor([test_input['attention_mask']])
input_id_tensor = torch.tensor([test_input['input_ids']]).squeeze(1)


# If you have a GPU, put everything on cuda
#input_id_tensor = input_id_tensor.to('cuda')
#mask_tensor = mask_tensor.to('cuda')
#model.to('cuda')

output=model(input_id_tensor,mask_tensor)
output[0]

tensor([[[10.0360, -0.9321, -1.7670, -1.4803, -1.7728, -1.2801, -2.0309,
          -1.3445, -1.4078],
         [ 9.7692, -1.1339, -1.9425, -1.5426, -1.7591, -0.8889, -2.0941,
          -1.4337, -1.5491],
         [ 9.3794, -1.3898, -1.4668, -1.9207, -1.2684, -1.6801, -1.8146,
          -2.0093,  0.1079],
         [-0.9720, -0.6946, -1.7232, -1.2652, -1.0235,  8.4143, -1.2986,
          -1.7473, -0.8502],
         [ 9.7465, -1.2631, -0.9779, -1.8997, -1.1748, -1.7780, -1.6329,
          -1.8663, -1.3235],
         [10.0416, -1.0232, -1.4028, -1.6850, -1.5457, -1.5152, -2.0047,
          -1.3275, -1.6532],
         [10.0943, -1.1116, -1.3706, -1.8046, -1.4542, -1.5873, -1.9817,
          -1.5149, -1.4469],
         [ 9.9520, -0.8237, -1.3681, -1.5506, -1.5959, -1.5955, -2.0343,
          -1.4882, -1.5045],
         [-0.7934, -0.7743, -1.6537, -1.4380, -1.1008,  8.4011, -1.2090,
          -1.6240, -0.9832],
         [ 9.9328, -1.2386, -1.2028, -1.8352, -1.2928, -1.7447, -1.7148,
         

We need to extract the logits from the output tensor

In [32]:
logits=output[0][0].detach().numpy()
logits

array([[10.035954  , -0.93205905, -1.7669843 , -1.4803059 , -1.7728072 ,
        -1.2801173 , -2.0308807 , -1.3445048 , -1.4078145 ],
       [ 9.769187  , -1.1339246 , -1.94248   , -1.5425957 , -1.7591249 ,
        -0.8889473 , -2.094071  , -1.4336938 , -1.5490689 ],
       [ 9.379381  , -1.3898025 , -1.4668384 , -1.9206967 , -1.2684286 ,
        -1.6800692 , -1.8145802 , -2.0093226 ,  0.10794801],
       [-0.9720163 , -0.69458985, -1.7231908 , -1.2651886 , -1.0234587 ,
         8.414303  , -1.2986304 , -1.7472861 , -0.85024023],
       [ 9.746473  , -1.263087  , -0.9778637 , -1.8997092 , -1.174839  ,
        -1.7780201 , -1.632939  , -1.8663412 , -1.3235496 ],
       [10.041648  , -1.0232471 , -1.4027673 , -1.6850045 , -1.5457171 ,
        -1.51521   , -2.0047445 , -1.3275328 , -1.6531973 ],
       [10.094293  , -1.1115838 , -1.3706303 , -1.8045654 , -1.4541761 ,
        -1.5873461 , -1.9817184 , -1.5149415 , -1.4468997 ],
       [ 9.951966  , -0.8237258 , -1.3681402 , -1.5505774 , -1

Then we can use numpy to get the most likely label according to the logits

In [33]:
predictions=np.argmax(logits, axis=-1)
predictions

array([0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0])

### Exercise 4
Write code to display the input id, true label id and predicted label id with one token per line e.g.:

```
101 : -100 : 0
4715 : 0 : 0
1011 : 0 : 0
2900 : 5 : 5
....
```

Can you improve this so that special tokens are removed and so that it shows the actual token, the actual true label and the actual predicted label e.g.:

```
soccer : O : O
- : O : O
japan : B-LOC : B-LOC
```



In [34]:
for input_id, label, pred in zip(test_input["input_ids"],test_input["labels"],predictions):
    print("{} : {} : {}".format(input_id,label,pred))

101 : -100 : 0
4715 : 0 : 0
1011 : 0 : 0
2900 : 5 : 5
2131 : 0 : 0
5341 : 0 : 0
2663 : 0 : 0
1010 : 0 : 0
2859 : 1 : 5
1999 : 0 : 0
4474 : 0 : 0
4154 : 0 : 0
1012 : 0 : 0
102 : -100 : 0


In [35]:
for input_id, label, pred in zip(test_input["input_ids"],test_input["labels"],predictions):
    
    if label !=-100:
        
        print("{} : {} : {}".format(tokenizer.convert_ids_to_tokens([input_id])[0],id2label[label],id2label[pred]))

soccer : O : O
- : O : O
japan : B-LOC : B-LOC
get : O : O
lucky : O : O
win : O : O
, : O : O
china : B-PER : B-LOC
in : O : O
surprise : O : O
defeat : O : O
. : O : O


### Exercise 5
Write some code which will run a list of sentences (e.g., from the test data) through the named entity recogniser and collect the predictions.  Compute the overall accuracy of the model on the test data

In [36]:


from tqdm import tqdm

def compute_metrics(allrefs,allpreds):
    print("Computing evaluation metrics")
    true_labels=[[label_names[l] for l in label if l!=-100] for label in allrefs]
    true_predictions = [[label_names[p] for (p,l) in zip(prediction,label) if l!=-100] for prediction,label in zip(allpreds,allrefs)]
    
    #we could compute other metrics as well, but here, for simplicity we are just going to compute overall accuracy
    accuracy = compute_accuracy(predictions=true_predictions,references=true_labels)
    return {"overall accuracy":accuracy}
    

def run_eval():
    allpreds=[]
    allrefs=[]
    count=0
    for test_input in tqdm(tokenized_datasets["test"]):
        count+=1
        mask=torch.tensor([test_input['attention_mask']])
        input_id = torch.tensor([test_input['input_ids']]).squeeze(1)
        output=model(input_id,mask)
        logits=output[0][0].detach().numpy()
        predictions=np.argmax(logits, axis=-1)
        allrefs.append(test_input['labels'])
        allpreds.append(predictions)
        if count %100==0:
            print(compute_metrics(allrefs,allpreds))

    print(compute_metrics(allrefs,allpreds))



    

In [37]:
run_eval()

  3%|█▏                                      | 101/3453 [00:09<03:50, 14.57it/s]

Computing evaluation metrics
{'overall accuracy': 0.9967637540453075}


  6%|██▎                                     | 201/3453 [00:19<07:09,  7.57it/s]

Computing evaluation metrics
{'overall accuracy': 0.9912229373902867}


  9%|███▍                                    | 301/3453 [00:29<05:10, 10.14it/s]

...

Computing evaluation metrics
{'overall accuracy': 0.9783885783458539}


 98%|██████████████████████████████████████▍| 3401/3453 [05:25<00:05, 10.32it/s]

Computing evaluation metrics
{'overall accuracy': 0.9779795925523336}


100%|███████████████████████████████████████| 3453/3453 [05:30<00:00, 10.44it/s]

Computing evaluation metrics
{'overall accuracy': 0.9780919825978869}


### Extension
Compute precision, recall and F1 for each of the different named entity labels.  Compute micro and macro-average F1 scores.